# 15 · Exporter comparison: analysis of the board logs *(no data, no retraining)*

Run after flashing the variants built by notebook 11 and saving each Serial output as `esp32/serial_logs/toolchain_<variant>.txt`
(for example `toolchain_prod_ours.txt`, `toolchain_prod_emlearn.txt`). Combines the logs with the host check written by notebook 11
(`results/esp32_export/<dataset>/toolchain_compare/toolchain_host_check.csv`) and produces

* `paper/tables/tc_toolchain_comparison.csv` (and a LaTeX version),
* `paper/figures/tc_tradeoff.png|pdf`,
* a Pareto summary over MCC, latency and firmware size.

Variants without a log are listed as not flashed.

In [ ]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "paper_utils.py").exists():
    if ROOT.parent == ROOT: raise RuntimeError("Open this notebook from inside the paper_package folder.")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display
import paper_utils as pu
DS = os.environ.get("ACTIVE_DATASET", "nf_ton_iot_v2")
BASE = ROOT / "results" / "esp32_export" / DS / "toolchain_compare"; LOGS = ROOT / "esp32" / "serial_logs"
FIG, TAB = ROOT / "paper" / "figures", ROOT / "paper" / "tables"; FIG.mkdir(parents=True, exist_ok=True); TAB.mkdir(parents=True, exist_ok=True)
hostf = BASE / "toolchain_host_check.csv"
print("dataset:", DS, "| host check:", hostf, "| exists:", hostf.exists())
assert hostf.exists(), "run notebook 11 first:  python run_all.py --mode quick --steps 11 --datasets " + DS
res = pd.read_csv(hostf); print("variants built by notebook 11:", list(res.variant))

## Which variants have a board log?

In [ ]:
hw, status = [], []
for name in res.variant:
    f = LOGS / f"toolchain_{name}.txt"
    if not f.exists(): status.append((name, "NOT FLASHED YET -> save the Serial output as " + f.name)); continue
    log = pu.parse_serial_log(f)
    if log["samples"].empty: status.append((name, "log found but no sample rows parsed (copy the WHOLE Serial output, including the per-sample lines)")); continue
    if log["legacy"]: status.append((name, "log is in the old format (flash the sketch generated by notebook 11)")); continue
    if not log["complete"]: status.append((name, "log is incomplete (no #DONE / SUMMARY line): wait for the run to finish before copying")); continue
    r = pu.summarize_hardware(DS, log); r["variant"] = name; hw.append(r); status.append((name, f"OK  ({len(log['samples'])} samples)"))
display(pd.DataFrame(status, columns=["variant", "status"]))

## Table and figure

In [ ]:
if hw:
    H = pd.DataFrame(hw)
    for c in H.columns:
        if c not in ("variant", "dataset", "log_format", "chip"): H[c] = pd.to_numeric(H[c], errors="coerce")
    keep = ["variant", "python_esp32_matches", "n_samples", "agreement", "mean_latency_us_batch", "median_us", "p95_us", "p99_us", "max_us", "benchmark_firmware_bytes", "flash_used_pct", "flash_headroom_bytes"]
    T = res.merge(H[[c for c in keep if c in H.columns]], on="variant", how="left")
    ref = T.loc[T.variant == "prod_ours", "benchmark_firmware_bytes"]
    if len(ref) and pd.notna(ref.iloc[0]): T["firmware_delta_vs_prod_ours_bytes"] = T["benchmark_firmware_bytes"] - float(ref.iloc[0])
    T.to_csv(TAB / "tc_toolchain_comparison.csv", index=False); display(T.round(3))
    pu.save_table(T[[c for c in ["variant", "group", "test_mcc", "agreement", "mean_latency_us_batch", "p95_us", "benchmark_firmware_bytes", "firmware_delta_vs_prod_ours_bytes"] if c in T.columns]],
                  TAB / "tc_toolchain_comparison_paper", "Toolchain and model-family comparison on the ESP32 (same preprocessing, same vectors).", "tab:toolchain", "{:.3f}")
    pu.set_paper_style(); good = T.dropna(subset=["mean_latency_us_batch"])
    fig, ax = plt.subplots(figsize=(7.5, 4.6))
    for _, r in good.iterrows():
        ax.scatter(r["mean_latency_us_batch"], r["test_mcc"], s=60 + 0.00018 * float(r["benchmark_firmware_bytes"]), alpha=.7, color="#1f77b4" if str(r["group"]).startswith("A") else "#d95f02")
        ax.annotate(r["variant"], (r["mean_latency_us_batch"], r["test_mcc"]), fontsize=7, xytext=(4, 4), textcoords="offset points")
    ax.set_xscale("log"); ax.set_xlabel("mean on-device latency per prediction (µs, log)"); ax.set_ylabel("test MCC")
    ax.set_title("accuracy vs latency (bubble = firmware size);  blue = toolchains, orange = model families")
    pu.save_fig(fig, FIG / "tc_tradeoff"); plt.show()
    print("saved:", TAB / "tc_toolchain_comparison.csv")
else:
    print("No usable board logs yet -> nothing to tabulate. Fix the items marked above, then re-run this notebook.")
    print("(The host-only result is already in", hostf, ")")

## Pareto summary

In [ ]:
def dominated(r, T):
    for _, o in T.iterrows():
        if o["variant"] == r["variant"]: continue
        ge = (o["test_mcc"] >= r["test_mcc"] - 1e-9) and (o["mean_latency_us_batch"] <= r["mean_latency_us_batch"]) and (o["benchmark_firmware_bytes"] <= r["benchmark_firmware_bytes"])
        gt = (o["test_mcc"] > r["test_mcc"] + 1e-9) or (o["mean_latency_us_batch"] < r["mean_latency_us_batch"]) or (o["benchmark_firmware_bytes"] < r["benchmark_firmware_bytes"])
        if ge and gt: return o["variant"]
    return None

if hw and T["mean_latency_us_batch"].notna().sum() >= 2:
    G = T.dropna(subset=["mean_latency_us_batch", "benchmark_firmware_bytes"]).copy()
    print("Pareto analysis over (higher MCC, lower latency, smaller firmware):\n")
    for _, r in G.iterrows():
        dom = dominated(r, G); print(f"  {r['variant']:20s} {'PARETO-OPTIMAL' if dom is None else 'dominated by ' + dom}")
    A = G[G.group.astype(str).str.startswith("A")]
    if "prod_ours" in set(A.variant) and len(A) > 1:
        o = A[A.variant == "prod_ours"].iloc[0]
        for _, r in A[A.variant != "prod_ours"].iterrows():
            print(f"\n  vs {r['variant']}: FaithTree export is {o['mean_latency_us_batch'] / r['mean_latency_us_batch'] - 1:+.1%} latency and {(o['benchmark_firmware_bytes'] - r['benchmark_firmware_bytes']) / 1024:+.1f} KB firmware relative to it; agreement {o['agreement']:.4f} vs {r['agreement']:.4f}")
else:
    print("Need logs for at least two variants (e.g. prod_ours and prod_emlearn) for a comparison.")